In [ ]:
import sympy as sp

In [ ]:
# Define Taylor expansion of sin
def Taylor_sin(x, n):
    result = 0
    for i in range((n-1)//2+1):
        result += (-1)**i * x**(2*i + 1) / sp.factorial(2*i + 1)
    return result

def tuple_to_sym(sym_tuple, number_tuple):
    res=1
    for i in range(len(sym_tuple)):
        res*=sym_tuple[i]**number_tuple[i]
    return res

x = sp.symbols('x')
sine_expanded = Taylor_sin(x,3)-x
a, a_dag, b, b_dag = sp.symbols("a a_dag b b_dag")
Z_a, Z_b, phi_q = sp.symbols("Z_a Z_b \phi_q")
phi_a, phi_b = sp.symbols("\\varphi_a \\varphi_b")
sine_expanded = sine_expanded.subs({x : (a + a_dag) + (b + b_dag)})

In [ ]:
sp.expand(sine_expanded)

In [ ]:
x_power_expanded = (x**7).subs({x : (a + a_dag)/sp.sqrt(2) + (b + b_dag)/sp.sqrt(2)})
expr_keep = sp.expand(x_power_expanded)

In [ ]:
polynom_vars = (a,b,a_dag,b_dag)
polynom_dict = sp.Poly(sine_expanded.expand(),polynom_vars).as_dict()
polynom_keep = {}
for k,v in polynom_dict.items():
    if (k[0]-k[2] - (k[3]-k[1])*2 == 0) and (k[0]>=2) and (k[3]==1):  #term resonant with the pump drive
            polynom_keep[k] = v
    #if k[0]-k[2] - (k[3]-k[1])*2 == -11:  
    #    polynom_keep[k] = v
    #elif k[0]-k[2] - (k[3]-k[1])*2 == 11:  
    #    polynom_keep[k] = v
    #elif k[0]==0 and k[2] == 0 :  #only b
    #    polynom_keep[k] = v
    #elif k[1]==0 and k[3] == 0 :  #only a
    #    polynom_keep[k] = v
    #if k[0]-k[2] == 0 or k[3]-k[1] == 0 : polynom_keep[k] = v

In [ ]:
polynom_keep

In [ ]:
import numpy as np
k_arr=np.array([0,1,2,3,4,5])
from scipy.special import factorial
factorial(k_arr)*factorial(k_arr+2)

In [ ]:
import json
polynom_json = {}
for k,v in polynom_dict.items():
    polynom_json[str(k)] = str(v.evalf())

with open("output.json", "w") as f:
    json.dump(polynom_json, f, indent=4)


In [ ]:
expr_keep = sum(v*tuple_to_sym(polynom_vars,k) for k,v in polynom_keep.items())

In [ ]:
expr_keep

In [ ]:
n_sub = 7
sub_dict = {sp.expand(((a + a_dag)/sp.sqrt(2))**n_sub) : (phi_a)**n_sub,
             sp.expand(((b + b_dag)/sp.sqrt(2))**n_sub) : (phi_b)**n_sub}
expr_keep = expr_keep.subs(sub_dict)
expr_keep

In [ ]:
eq_a = sp.simplify(sp.diff(expr_keep, a_dag))
sub_dict = {sp.expand(((a + a_dag))**4) : (a+a_dag)**4,
             sp.expand(((b + b_dag))**4) : (b+b_dag)**4,
             sp.expand(a_dag*((b + b_dag))**3) : a_dag*(b+b_dag)**3}
eq_a_rest = (eq_a*4/5*sp.sqrt(2))+sp.expand(-4*a_dag*(b+b_dag)**3 - (b+b_dag)**4 - 4*a**3*(b+b_dag))
eq_a_rest/2
#((eq_a*8/5*sp.sqrt(2)/2).subs(sub_dict)/4).subs(sub_dict)*4

In [ ]:
q_a, q_b = sp.symbols("q_a q_b")
sub_dict = {a:(phi_a+sp.I*q_a)/sp.sqrt(2), b:(phi_b+sp.I*q_b)/sp.sqrt(2), a_dag:(phi_a-sp.I*q_a)/sp.sqrt(2), b_dag:(phi_b-sp.I*q_b)/sp.sqrt(2)}
expr_keep = sp.expand(expr_keep.subs(sub_dict))
expr_keep

In [ ]:
x_power_expanded = (x**4).subs({x : sp.sqrt(Z_a)*(a + a_dag)/sp.sqrt(2)/phi_q + sp.sqrt(Z_b)*(b + b_dag)/sp.sqrt(2)/phi_q})

In [ ]:
sub_dict = {sp.expand((sp.sqrt(Z_a)*(a + a_dag)/sp.sqrt(2)/phi_q)**4) : (sp.sqrt(Z_a)*phi_a/phi_q)**4, sp.expand((sp.sqrt(Z_b)*(b + b_dag)/sp.sqrt(2)/phi_q)**4) : (sp.sqrt(Z_b)*phi_b/phi_q)**4}
expr = sp.simplify(sp.expand(x_power_expanded).subs(sub_dict))
expr

In [ ]:
x_power_expanded = (x**4).subs({x : (a + a_dag)/sp.sqrt(2) + (b + b_dag)/sp.sqrt(2)})

In [ ]:
sub_dict = {sp.expand(((a + a_dag)/sp.sqrt(2))**4) : (phi_a)**4, sp.expand(((b + b_dag)/sp.sqrt(2))**4) : (phi_b)**4}
expr = sp.simplify(sp.expand(x_power_expanded).subs(sub_dict))
expr

In [ ]:
x_power_expanded = (x**4).subs({x : sp.sqrt(Z_a)*phi_a/phi_q + sp.sqrt(Z_b)*phi_b/phi_q})

In [ ]:
sp.expand(x_power_expanded)

In [ ]:
np.sqrt(0.11)*1.23

# Hamiltonian and harmonic balance

In [ ]:
E = sp.symbols("E")
I=sp.I
eps_d = sp.symbols("\\varepsilon_d")
tau = sp.symbols("\\tau")
kappa_b = sp.symbols("\\kappa_b")
phi_a, phi_b = sp.symbols("\\varphi_a \\varphi_b")
beta_a, beta_b = sp.symbols("\\beta_a \\beta_b")
a, a_dag, b, b_dag = sp.symbols("a a_dag b b_dag", complex=True)
a_0, b_0 = sp.symbols("a_0 b_0", complex=True)
a_1, b_1, b_2 = sp.symbols("a_1 b_1 b_2", real=True)
sine_expanded = Taylor_sin(x,3)
#H_coupling = -E*(sine_expanded.subs({x : phi_a*(a + a_dag) + phi_b*(b + b_dag)})-(phi_a*(a + a_dag) + phi_b*(b + b_dag)))
H_coupling = E*(a**2*b_dag/2+a_dag**2*b/2+(a+a_dag)*b*b_dag+(b+b_dag)*a*a_dag+a**2*a_dag/2+a_dag**2*a/2+ a*b_dag**2/2+a_dag*b**2/2+b*b_dag**2/2+b_dag*b**2/2)
H_coupling = H_coupling.subs(a,phi_a*a).subs(a_dag,phi_a*a_dag).subs(b,phi_b*b).subs(b_dag,phi_b*b_dag)
H_coupling_R = H_coupling.subs({a : a*sp.exp(-sp.I*tau), a_dag : a_dag*sp.exp(sp.I*tau), b : b*sp.exp(-2*sp.I*tau), b_dag : b_dag*sp.exp(2*sp.I*tau) })
H_R = eps_d*(sp.exp(2*sp.I*tau)+sp.exp(-2*sp.I*tau))/2*(b*sp.exp(-2*sp.I*tau)+b_dag*sp.exp(2*sp.I*tau))+H_coupling_R

In [ ]:
H_R

In [ ]:
eq_a = sp.expand(-sp.I*sp.diff(H_R,a_dag)).subs(E*phi_a,beta_a).subs(phi_a*a,a).subs(phi_a*a_dag,a_dag).subs(phi_b*b,b).subs(phi_b*b_dag,b_dag)
eq_b = sp.expand(-sp.I*sp.diff(H_R,b_dag)-kappa_b*b).subs(E*phi_b,beta_b).subs(phi_a*a,a).subs(phi_a*a_dag,a_dag).subs(phi_b*b,b).subs(phi_b*b_dag,b_dag)

In [ ]:
eq_a

In [ ]:
eps = sp.Symbol('eps')
sub_dict = {a : (a_0+eps*a_1*sp.exp(sp.I*tau)),
            a_dag : (sp.conjugate(a_0)+eps*sp.conjugate(a_1)*sp.exp(-sp.I*tau)), 
            b : (b_0+eps*b_1*sp.exp(sp.I*tau)+eps*b_2*sp.exp(2*sp.I*tau)), 
            b_dag : (sp.conjugate(b_0)+eps*sp.conjugate(b_1)*sp.exp(-sp.I*tau)+eps*sp.conjugate(b_2)*sp.exp(-2*sp.I*tau))}
eq_a_four = sp.expand(eq_a.subs(sub_dict))
eq_b_four = sp.expand(eq_b.subs(sub_dict))

eq_a_four_lin = sp.series(sp.expand(eq_a_four), eps, 0, 2).removeO().subs(eps, 1)
eq_b_four_lin = sp.series(sp.expand(eq_b_four), eps, 0, 2).removeO().subs(eps, 1)

In [ ]:
eq_a_four_lin

In [ ]:
def exp_fourier_coeffs(expr, tau):
    """
    Return dict {n: Cn} such that expr = sum_n Cn*exp(I*n*tau),
    assuming all tau-dependence is through exp(I*n*tau) with (near-)integer n.
    Handles float n by nsimplify -> int.
    """
    expr = sp.expand(expr)
    I=sp.I

    coeffs = {}

    for term in expr.as_ordered_terms():
        # Separate exp factors from the rest
        exp_factors = []
        other = term

        for f in sp.factor_terms(term).as_ordered_factors():
            if f.func == sp.exp and f.args[0].has(tau):
                exp_factors.append(f)
            else:
                # multiply non-exp factors into 'other'
                other = other / f if f in exp_factors else other

        # Easier: just pull exp(I*n*tau) out by searching
        # (works even if term is a product)
        n_total = 0
        rest = term

        for e in term.atoms(sp.exp):
            arg = sp.simplify(e.args[0])
            if arg.has(tau):
                # try to read n from exp(I*n*tau)
                n = sp.simplify(arg / (I*tau))
                n = sp.nsimplify(n)  # converts 5.0 -> 5, etc.
                if n.is_Integer:
                    n_total += int(n)
                    rest = sp.simplify(rest / sp.exp(I*int(n)*tau))
                else:
                    raise ValueError(f"Non-integer frequency found: exp({arg}) gives n={n}")

        coeffs[n_total] = coeffs.get(n_total, 0) + sp.simplify(rest)
        res = {n: sp.simplify(c) for n, c in coeffs.items()}
    return {0 : res[0], 1 : res[1], 2 : res[2] }


In [ ]:
coeff_dict_a = exp_fourier_coeffs(eq_a_four_lin, tau)
coeff_dict_a


In [ ]:
coeff_dict_b = exp_fourier_coeffs(eq_b_four_lin, tau)
coeff_dict_b

We notice that $b_1=0$ since coeff_dict_a["0"]=0 and coeff_dict_b["1"] = coeff_dict_a["0"] - kappa_b*b_1 = b_1/2 and that $b_2 = \beta_b'/\beta_a' a_1$

In [ ]:
eq_a_four_lin_bis = eq_a_four_lin.subs(b_1, 0)
eq_b_four_lin_bis = eq_b_four_lin.subs(b_1, 0)

In [ ]:
coeff_dict_a = exp_fourier_coeffs(eq_a_four_lin_bis, tau)
coeff_dict_a

In [ ]:
coeff_dict_b = exp_fourier_coeffs(eq_b_four_lin_bis, tau)
coeff_dict_b

In [ ]:
import sympy as sp

# Define symbols
beta_a, beta_b, eps = sp.symbols(
    'beta_a beta_b varepsilon', real=True
)
b = 1/(12*(beta_a+beta_b))+sp.sqrt((1/(12*(beta_a+beta_b)))**2+eps/(3*beta_b))
gamma = 1+beta_b/beta_a
gamma_p = 1+2*beta_b/(3*beta_a)

# Define the expression
expr = (
    (1/(2*beta_a*gamma_p) - b)**2 * (1/(4*beta_b) - b)**2
    - 2*(1/(2*beta_a*gamma_p) - b)*(1/(4*beta_b) - b) * b/(2*beta_a*gamma)
    - (1/(4*beta_b) - b)**2 * (-2*b**2 + eps/beta_b)
    + 4*b*(1/(4*beta_b) - b) * (-2*b**2 + eps/beta_b)
    + (-3*b**2 + eps/beta_b)**2
)

expr

In [ ]:
# Expand and simplify
expanded = sp.expand(expr)

In [ ]:
expanded

In [ ]:
import numpy as np

eps_p = np.linspace(0,0.55*np.pi,100)
beta_a, beta_b = 0.1831084342479521, 0.3148859751926766 
eps = 0.009779030042836782

b = 1/(12*(beta_a+beta_b))+np.sqrt((1/(12*(beta_a+beta_b)))**2+eps/(3*beta_b))
gamma = 1+beta_b/beta_a
gamma_p = 1+2*beta_b/(3*beta_a)

# Define the expression
expr = (
    (1/(2*beta_a*gamma_p) - b)**2 * (1/(4*beta_b) - b)**2
    - 2*(1/(2*beta_a*gamma_p) - b)*(1/(4*beta_b) - b) * b/(2*beta_a*gamma)
    - (1/(4*beta_b) - b)**2 * (-2*b**2 + eps/beta_b)
    + 4*b*(1/(4*beta_b) - b) * (-2*b**2 + eps/beta_b)
    + (-3*b**2 + eps/beta_b)**2
)

In [ ]:
expr